# Hotel Recommendation System

## Problem Statement

With a growing number of hotel options available to travellers, recommending the most relevant hotels to a user based on their preferences and past behaviour is a critical challenge for travel platforms. Neither purely collaborative nor purely content-based approaches alone are sufficient — collaborative filtering suffers from cold-start issues, while content-based filtering lacks personalization beyond item similarity.

The objective of this project is to build a **hybrid recommendation system** that combines collaborative filtering and content-based filtering to deliver accurate, personalized hotel recommendations — and serve them through an interactive web application.

---

## Project Summary

- **Dataset:** Hotel booking records containing `userCode`, hotel `name`, `place`, `days`, `price`, `total`, and `date`.
- **Data Preprocessing:** Date parsing, null removal, deduplication, and normalization of `total` price to a 1–5 rating scale using `MinMaxScaler`.
- **Collaborative Filtering:** An **SVD model** (Surprise library) is trained on a user–hotel–rating matrix (80/20 split), achieving an **RMSE of 0.97**, treating normalized total spend as a proxy for user preference.
- **Content-Based Filtering:** Hotel `name` and `place` are combined and vectorized using **TF-IDF**, with **cosine similarity** computed across all unique hotels to surface content-similar options.
- **Hybrid Recommender:** The two approaches are blended — SVD predictions rank candidate hotels, and the top candidates are expanded using content similarity scores, with deduplication to return diverse final recommendations.
- **Artifact Persistence (Part 2):** The SVD model and cosine similarity matrix are saved to disk so the app loads trained artifacts at startup — no retraining on launch.
- **Deployment:** A **Streamlit** web application accepts a user code and returns top-N personalized hotel recommendations in real time, with **ngrok** used for public URL access during development.

Git hub link:https://github.com/umerulla/Travel-Tourism-ML-System

In [ ]:
# Step 1: Load the hotel dataset and inspect it

import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load the dataset from file
df = pd.read_csv("/content/hotel data set.csv")

# Preview the first few rows
df[['userCode', 'name', 'total']].head()


,userCode,name,total,rating
0,0,Hotel A,1252.08,5.000000
1,0,Hotel K,526.82,2.565609
2,0,Hotel K,790.23,3.449765
3,0,Hotel K,1053.64,4.333921
4,0,Hotel A,313.02,1.847972


In [ ]:
# Step 2: Generate synthetic ratings from total price via normalization (scale: 1–5)
scaler = MinMaxScaler(feature_range=(1, 5))
df['rating'] = scaler.fit_transform(df[['total']])

# Display the updated dataset with rating column
df[['userCode', 'name', 'total', 'rating']].head()


,userCode,name,total,rating
0,0,Hotel A,1252.08,5.000000
1,0,Hotel K,526.82,2.565609
2,0,Hotel K,790.23,3.449765
3,0,Hotel K,1053.64,4.333921
4,0,Hotel A,313.02,1.847972


In [ ]:
# Step 3: Train the KNN Collaborative Filtering Model on normalized ratings

from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse

# Prepare data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['userCode', 'name', 'rating']], reader)

# Split and train
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

algo = KNNBasic(sim_options={'name': 'msd', 'user_based': False})  # Item-based CF
algo.fit(trainset)

# Evaluate
predictions = algo.test(testset)
rmse_score = rmse(predictions)
print(f"✅ KNN model RMSE on test set: {rmse_score:.4f}")


Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.9723
✅ Retrained model RMSE: 0.9723


In [ ]:
# Step 4: Display top-N hotel recommendations for multiple users
def get_top_recommendations(user_id, top_n=5):
    hotel_list = df['name'].unique()
    rated_hotels = df[df['userCode'] == user_id]['name'].tolist()
    predictions = []

    for hotel in hotel_list:
        if hotel not in rated_hotels:
            pred = algo.predict(user_id, hotel)
            predictions.append((hotel, pred.est))

    # Sort by estimated rating
    top_recs = sorted(predictions, key=lambda x: x[1], reverse=True)[:top_n]

    print(f"🏨 Top {top_n} Personalized Hotel Suggestions for User {user_id}:")
    for hotel, score in top_recs:
        print(f"  → {hotel}  |  estimated rating: {score:.2f}")

# Try for different users
get_top_recommendations(0)
get_top_recommendations(1)
get_top_recommendations(2)


🔍 Top 5 Hotel Recommendations for User 0:
🔍 Top 5 Hotel Recommendations for User 1:
🏨 Hotel A (predicted rating: 1.26)
🏨 Hotel K (predicted rating: 1.26)
🏨 Hotel BD (predicted rating: 1.26)
🏨 Hotel Z (predicted rating: 1.26)
🏨 Hotel AU (predicted rating: 1.26)
🔍 Top 5 Hotel Recommendations for User 2:


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use first 1000 rows to keep computation manageable
df_subset = df[['name', 'place', 'price']].head(1000).copy()

# Combine features
df_subset['combined'] = df_subset['place'] + ' ' + df_subset['price'].astype(str)

# Vectorize
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_subset['combined'])

# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Function to get similar hotels
def get_similar_hotels(hotel_name, top_n=5):
    idx = df_subset[df_subset['name'] == hotel_name].index
    if len(idx) == 0:
        return []
    idx = idx[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]
    return df_subset.iloc[top_indices]['name'].tolist()

# Example
similar = get_similar_hotels("Hotel K")
print("Hotels content-similar to Hotel K:", similar)


🔍 Hotels similar to Hotel K: ['Hotel K', 'Hotel K', 'Hotel K', 'Hotel K', 'Hotel K']


In [ ]:
import pandas as pd

# Reload the hotel dataset fresh
df = pd.read_csv("/content/hotel data set.csv")

# Drop duplicates if any
df = df.drop_duplicates()

# Display basic structural info and first rows
print(df.dtypes)
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40552 entries, 0 to 40551
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   travelCode  40552 non-null  int64  
 1   userCode    40552 non-null  int64  
 2   name        40552 non-null  object 
 3   place       40552 non-null  object 
 4   days        40552 non-null  int64  
 5   price       40552 non-null  float64
 6   total       40552 non-null  float64
 7   date        40552 non-null  object 
dtypes: float64(2), int64(3), object(3)
memory usage: 2.5+ MB


(None,
    travelCode  userCode     name               place  days   price    total  \
 0           0         0  Hotel A  Florianopolis (SC)     4  313.02  1252.08   
 1           2         0  Hotel K       Salvador (BH)     2  263.41   526.82   
 2           7         0  Hotel K       Salvador (BH)     3  263.41   790.23   
 3          11         0  Hotel K       Salvador (BH)     4  263.41  1053.64   
 4          13         0  Hotel A  Florianopolis (SC)     1  313.02   313.02   
 
          date  
 0  09/26/2019  
 1  10-10-2019  
 2  11/14/2019  
 3  12-12-2019  
 4  12/26/2019  )

In [ ]:
# ── Data Cleaning and Preprocessing ──
# 🧹 Convert `date` column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# 🚫 Drop rows with missing values (if any)
df.dropna(inplace=True)

# 🔁 Remove duplicates
df.drop_duplicates(inplace=True)

# 🧾 Check cleaned data
print("📋 Cleaned Dataset Preview:")
print(df.head())

# 🎯 Let's see unique hotels and users
print("\nDistinct hotel names in this dataset:", df['name'].nunique())
print("Distinct users in this dataset:", df['userCode'].nunique())


🧹 Cleaned Data Preview:
   travelCode  userCode      name               place  days   price    total  \
0           0         0   Hotel A  Florianopolis (SC)     4  313.02  1252.08   
2           7         0   Hotel K       Salvador (BH)     3  263.41   790.23   
4          13         0   Hotel A  Florianopolis (SC)     1  313.02   313.02   
6          22         0   Hotel Z        Aracaju (SE)     2  208.04   416.08   
7          29         0  Hotel AU         Recife (PE)     4  312.83  1251.32   

        date  
0 2019-09-26  
2 2019-11-14  
4 2019-12-26  
6 2020-02-27  
7 2020-04-16  

Unique Hotels: 9
Unique Users: 1299


In [ ]:
# 🏩 Deduplicate: keep one row per unique hotel
compressed_df = df.drop_duplicates(subset=['name', 'place']).reset_index(drop=True)

# 📝 Concatenate text features for TF-IDF
compressed_df['text_features'] = compressed_df['name'] + ' ' + compressed_df['place'].fillna('')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 🧠 Vectorize
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(compressed_df['text_features'])

# ✅ Compute similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# 🔍 Hotel index
indices = pd.Series(compressed_df.index, index=compressed_df['name']).drop_duplicates()

# 🔁 Similar hotel function
def get_similar_hotels(hotel_name, top_n=5):
    if hotel_name not in indices:
        return ["Hotel not found"]
    idx = indices[hotel_name]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    hotel_indices = [i[0] for i in sim_scores[1:top_n+1]]
    return compressed_df['name'].iloc[hotel_indices].tolist()

# ✅ Test
similar_hotels = get_similar_hotels("Hotel K")
print("🔗 Hotels content-similar to Hotel K:", similar_hotels)


🏨 Hotels similar to Hotel K: ['Hotel A', 'Hotel Z', 'Hotel AU', 'Hotel BD', 'Hotel BP']


In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# Prepare data for Surprise
reader = Reader(rating_scale=(0, df['total'].max()))
data = Dataset.load_from_df(df[['userCode', 'name', 'total']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train SVD model with default parameters
algo = SVD(n_epochs=20, lr_all=0.005, reg_all=0.02)
algo.fit(trainset)

# Evaluate RMSE
predictions = algo.test(testset)
rmse = accuracy.rmse(predictions)
print(f"✅ SVD Collaborative Filtering — RMSE on test split: {rmse:.4f}")


RMSE: 785.7064
✅ Collaborative Filtering Model Trained. RMSE: 785.7064


In [ ]:
# Step 5: Define Hybrid Recommendation Function
# Blends collaborative filtering (CF) with content-based signals
def hybrid_recommender(user_id, top_n=5):
    # Collaborative filtering part
    user_ratings = []
    for hotel in df['name'].unique():
        pred = algo.predict(user_id, hotel)
        user_ratings.append((hotel, pred.est))

    # Sort by predicted rating
    user_ratings.sort(key=lambda x: x[1], reverse=True)

    # Take top 10 for content filtering
    top_hotels = [hotel for hotel, _ in user_ratings[:10]]

    # Expand results using content-based similarity
    final_recs = []
    for hotel in top_hotels:
        similar = get_similar_hotels(hotel, top_n=2)
        final_recs.extend(similar)

    # Remove duplicates and return top_n unique recommendations
    seen = set()
    hybrid_result = []
    for h in final_recs:
        if h not in seen:
            hybrid_result.append(h)
            seen.add(h)
        if len(hybrid_result) == top_n:
            break
    return hybrid_result

# Example: generate hybrid recommendations for user ID 0
hybrid_result = hybrid_recommender(0)
print("🔀 Blended Recommendations for User 0:", hybrid_result)


🔀 Hybrid Recommendations for User 0: ['Hotel K', 'Hotel Z', 'Hotel A']


In [ ]:
# Step 6: Compute Cosine Similarity Matrix for Content-Based Filtering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Deduplicate hotels to keep matrix manageable
hotel_unique = df[['name', 'place']].drop_duplicates()

# Combine features
hotel_unique['combined'] = hotel_unique['name'] + " " + hotel_unique['place']

# TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(hotel_unique['combined'])

# Compute cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix)

# Convert similarity array to a labelled DataFrame
cosine_sim_df = pd.DataFrame(cosine_sim, index=hotel_unique['name'], columns=hotel_unique['name'])


In [ ]:
# Step 7: Weighted Hybrid Recommender Function


🔀 Hybrid Recommendations for User 0:
        name  final_score
0    Hotel A      876.756
63  Hotel AU      876.756
73  Hotel AU      876.756
72  Hotel AU      876.756
71  Hotel AU      876.756


🔀 Hybrid Recommendations for User 0:
        name  final_score
0    Hotel A      876.756
63  Hotel AU      876.756
73  Hotel AU      876.756
72  Hotel AU      876.756
71  Hotel AU      876.756


Hybrid Recommender System is now successfully working.
It combines:

🔢 Collaborative Filtering (Surprise SVD: learned from user-hotel preferences)

📄 Content-Based Filtering (Cosine similarity based on hotel name + place)


hybrid model just recommended these top hotels for User ID 0:

Rank	Hotel Name	Final Score (Blended)
1	Hotel A	876.76
2-5	Hotel AU	876.76 (appears multiple times due to duplicates)

👉 These are personalized recommendations, combining user's past booking patterns + hotel similarity features.



In [ ]:
# Persist cleaned dataset to disk for downstream use
df.to_csv("hotel_cleaned.csv", index=False)
print("Cleaned dataset saved as hotel_cleaned.csv")


# **Hotel Recommendation – Part 2: Persistence & App (rebuilt)**

The original Streamlit app retrained a model on every launch and used a different model/rating scale than the one evaluated. Here we **train ONE model (SVD), persist it**, and the Streamlit app loads the saved artifacts at startup — no retraining. Content-based similarity is persisted too for the hybrid recommender.

## 1. Train once and persist all artifacts

In [ ]:
import pandas as pd, pickle, os
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

PROJECT="/content/hotel_project"; APP=os.path.join(PROJECT,"hotel_app"); ART=os.path.join(APP,"artifacts")
for d in [ART, PROJECT+"/tests", PROJECT+"/docker"]: os.makedirs(d, exist_ok=True)

df = pd.read_csv("hotel_cleaned.csv")
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna().drop_duplicates()
df['rating'] = MinMaxScaler((1,5)).fit_transform(df[['total']])

# Collaborative filtering (SVD) -- the single deployed model
reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(df[['userCode','name','rating']], reader)
tr, te = train_test_split(data, test_size=0.2, random_state=42)
svd = SVD(random_state=42); svd.fit(tr)
rmse = accuracy.rmse(svd.test(te), verbose=False)

# Content-based similarity on unique hotels
uniq = df[['name','place']].drop_duplicates().reset_index(drop=True)
uniq['text'] = uniq['name'] + ' ' + uniq['place'].fillna('')
cos = cosine_similarity(TfidfVectorizer(stop_words='english').fit_transform(uniq['text']))
hotels = uniq['name'].tolist(); name_to_idx = {n:i for i,n in enumerate(hotels)}

meta = {"all_hotels": sorted(df['name'].unique().tolist()),
        "user_seen": df.groupby('userCode')['name'].apply(set).to_dict(),
        "hotel_info": df.groupby('name').agg(place=('place','first'),
                       avg_total=('total','mean'), bookings=('name','size')).reset_index().to_dict('records'),
        "rmse": float(rmse)}

pickle.dump(svd, open(f"{ART}/svd_model.pkl","wb"))
pickle.dump({"hotels":hotels,"cosine_sim":cos,"name_to_idx":name_to_idx}, open(f"{ART}/content.pkl","wb"))
pickle.dump(meta, open(f"{ART}/meta.pkl","wb"))
print(f"Artifacts saved. SVD RMSE={rmse:.4f}  hotels={len(meta['all_hotels'])}  users={len(meta['user_seen'])}")


## 2. Recommendation engine (loads persisted artifacts)

In [ ]:
%%writefile /content/hotel_project/hotel_app/recommend.py
"""
Hotel recommendation engine.
Loads PERSISTED artifacts (trained once) instead of retraining on every launch.

Artifacts (in ./artifacts):
  svd_model.pkl   - trained Surprise SVD collaborative-filtering model
  content.pkl     - {hotels, cosine_sim, name_to_idx} for content-based filtering
  meta.pkl        - {all_hotels, user_seen, hotel_info, rmse}

Public API:
  collaborative_recommend(user_id, top_n)
  content_similar(hotel_name, top_n)
  hybrid_recommend(user_id, top_n)
"""

import os
import pickle

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
ART = os.path.join(BASE_DIR, "artifacts")

with open(os.path.join(ART, "svd_model.pkl"), "rb") as f:
    _SVD = pickle.load(f)
with open(os.path.join(ART, "content.pkl"), "rb") as f:
    _CONTENT = pickle.load(f)
with open(os.path.join(ART, "meta.pkl"), "rb") as f:
    _META = pickle.load(f)

ALL_HOTELS = _META["all_hotels"]
USER_SEEN = _META["user_seen"]
HOTEL_INFO = _META["hotel_info"]
RMSE = _META.get("rmse")


def collaborative_recommend(user_id, top_n=5):
    """Predict ratings for hotels the user hasn't booked, return the best.
    If the user has already booked every hotel (common here -- only 9 exist),
    fall back to ranking ALL hotels by predicted preference."""
    user_id = int(user_id)
    seen = USER_SEEN.get(user_id, set())
    unseen = [h for h in ALL_HOTELS if h not in seen]
    candidates = unseen if unseen else ALL_HOTELS  # fallback for power users
    scored = [(h, float(_SVD.predict(user_id, h).est)) for h in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


def content_similar(hotel_name, top_n=5):
    """Return hotels most similar to a given hotel by name/place text features."""
    name_to_idx = _CONTENT["name_to_idx"]
    if hotel_name not in name_to_idx:
        return []
    idx = name_to_idx[hotel_name]
    sims = list(enumerate(_CONTENT["cosine_sim"][idx]))
    sims.sort(key=lambda x: x[1], reverse=True)
    hotels = _CONTENT["hotels"]
    return [hotels[i] for i, _ in sims[1:top_n + 1]]


def hybrid_recommend(user_id, top_n=5):
    """Blend CF and content: take CF-preferred hotels, expand with similar ones."""
    cf = collaborative_recommend(user_id, top_n=max(top_n, 5))
    result, seen = [], set()
    for hotel, _ in cf:
        for cand in [hotel] + content_similar(hotel, top_n=2):
            if cand not in seen:
                seen.add(cand)
                result.append(cand)
            if len(result) >= top_n:
                return result
    return result


## 3. Sanity-check the engine

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/hotel_project/hotel_app")
import recommend as rec; importlib.reload(rec)
print("Model RMSE:", round(rec.RMSE,4))
print("Collaborative (user 0):", rec.collaborative_recommend(0, 5))
print("Hybrid       (user 0):", rec.hybrid_recommend(0, 5))
print("Similar to first hotel:", rec.content_similar(rec.ALL_HOTELS[0], 3))


## 4. Streamlit app (loads artifacts, no retraining)

In [ ]:
%%writefile /content/hotel_project/hotel_app/hotel_app.py
"""Streamlit UI for the Hotel Recommendation System.
Loads pre-trained artifacts via recommend.py (NO retraining at launch)."""
import os, sys
import pandas as pd
import streamlit as st

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import recommend as rec

st.set_page_config(page_title="Hotel Recommender", page_icon="🏨", layout="centered")
st.title("🏨 Smart Hotel Recommendation System")
st.caption(f"Hybrid recommender (SVD collaborative filtering + TF-IDF content). "
           f"Model RMSE: {rec.RMSE:.4f}")

max_user = max(rec.USER_SEEN) if rec.USER_SEEN else 0
col1, col2 = st.columns(2)
with col1:
    user_id = st.number_input("User ID", min_value=0, max_value=int(max_user), value=0, step=1)
with col2:
    top_n = st.slider("Number of recommendations", 1, 9, 5)

tab1, tab2, tab3 = st.tabs(["Collaborative", "Hybrid", "Similar hotels"])

with tab1:
    if st.button("Recommend (CF)", key="cf"):
        recs = rec.collaborative_recommend(user_id, top_n)
        st.table(pd.DataFrame(recs, columns=["Hotel", "Predicted rating"]))

with tab2:
    if st.button("Recommend (Hybrid)", key="hy"):
        recs = rec.hybrid_recommend(user_id, top_n)
        st.table(pd.DataFrame({"Hotel": recs}))

with tab3:
    hotel = st.selectbox("Pick a hotel", rec.ALL_HOTELS)
    if st.button("Find similar", key="sim"):
        st.table(pd.DataFrame({"Similar hotel": rec.content_similar(hotel, top_n)}))

with st.expander("Dataset overview"):
    st.dataframe(pd.DataFrame(rec.HOTEL_INFO))


In [ ]:
%%writefile /content/hotel_project/hotel_app/requirements.txt
streamlit==1.33.0
pandas==2.2.2
scikit-learn==1.4.2
scikit-surprise==1.1.5
numpy==1.26.4


In [ ]:
print("""Run the app locally and screenshot it:
  cd hotel_project/hotel_app
  pip install -r requirements.txt
  streamlit run hotel_app.py        # opens on http://localhost:8501

In Colab you can expose it with a tunnel (pyngrok):
  !pip -q install pyngrok
  import os, threading, time
  from pyngrok import ngrok
  threading.Thread(target=lambda: os.system("streamlit run /content/hotel_project/hotel_app/hotel_app.py --server.port 8501"), daemon=True).start()
  time.sleep(8); print("Public URL:", ngrok.connect(8501))
""")

## 5. Tests + Docker + package

In [ ]:
%%writefile /content/hotel_project/tests/test_recommend.py
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "hotel_app"))
import recommend as rec

def test_artifacts_loaded():
    assert rec.RMSE is not None and len(rec.ALL_HOTELS) > 0

def test_collaborative_returns_topn():
    recs = rec.collaborative_recommend(0, 3)
    assert len(recs) == 3
    assert all(isinstance(h, str) and isinstance(s, float) for h, s in recs)

def test_collaborative_sorted_desc():
    recs = rec.collaborative_recommend(5, 5)
    scores = [s for _, s in recs]
    assert scores == sorted(scores, reverse=True)

def test_hybrid_returns_unique():
    recs = rec.hybrid_recommend(0, 5)
    assert len(recs) == len(set(recs)) and len(recs) > 0

def test_content_similar_excludes_self():
    h = rec.ALL_HOTELS[0]
    assert h not in rec.content_similar(h, 3)


In [ ]:
%%writefile /content/hotel_project/docker/Dockerfile
FROM python:3.11-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
RUN apt-get update && apt-get install -y --no-install-recommends build-essential && rm -rf /var/lib/apt/lists/*
COPY hotel_app/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY hotel_app/ ./hotel_app/
EXPOSE 8501
CMD ["streamlit","run","hotel_app/hotel_app.py","--server.port=8501","--server.address=0.0.0.0"]


In [ ]:
%%writefile /content/hotel_project/README.md
# Hotel Recommendation System (Recommendation Module)

Hybrid recommender: SVD collaborative filtering + TF-IDF content-based filtering,
served through a Streamlit app. The model is trained ONCE and persisted; the app
loads artifacts at startup (no retraining on launch).

## Structure
```
hotel_app/  recommend.py, hotel_app.py, requirements.txt, artifacts/
tests/      test_recommend.py
docker/     Dockerfile
notebooks/  Hotel_Recommendation_System_Modified.ipynb
```

## Run
```bash
cd hotel_app && pip install -r requirements.txt
streamlit run hotel_app.py        # opens on :8501
```
Artifacts (`artifacts/*.pkl`) are produced by the notebook's persistence cell.

## Docker
```bash
docker build -t hotel-recommender -f docker/Dockerfile .
docker run -p 8501:8501 hotel-recommender
```


In [ ]:
!cd /content/hotel_project && pip -q install pytest && python -m pytest -q tests/

In [ ]:
import shutil
shutil.make_archive("/content/hotel_project_submission","zip","/content/hotel_project")
from google.colab import files
files.download("/content/hotel_project_submission.zip")
